# 04_analyze_eleveld_outputs

This notebook performs the primary regression and figure-generation workflow for the Eleveld-based induction analysis.

It loads the patient-level simulation output, applies the primary analytic cohort restrictions, models age-associated changes in weight-normalised induction dose and peak modelled effect-site concentration, derives the age-adjusted $Ce_{50}$ reference and model-predicted required dose, and generates the main manuscript figure and summary tables.

## Outputs

- Figure 1
- Table S2

## Notes

- $C_{e,\max}$ is model-derived.
- The required-dose calculation is a proportional scaling estimate based on the observed dosing pattern.
- Private source data and institution-specific paths are omitted from the public notebook.

## Configuration and Imports
This section defines the input file used for the primary analysis and loads the Python packages required for modeling and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import bs
from statsmodels.stats.proportion import proportion_confint


# This file should be the .pkl output generated in Notebook 03 (Simulation)
INPUT_FILE = "INSERT_SIMULATION_RESULTS_FILE.pkl"



## Load Eleveld Simulation Output
Loads the patient-level dataset generated by the simulation engine.

In [ ]:
df_analysis = pd.read_pickle(INPUT_FILE)

print(f"Loaded {len(df_analysis):,} rows from {INPUT_FILE}.")

## Apply primary analytic cohort restrictions

This section applies the age-based cohort restrictions used in the main analysis and reports cohort attrition for reproducibility and manuscript flow reporting.

In [ ]:
# 1. PREPARATION (18-89 Only with Attrition Reporting)
# ---------------------------------------------------
df = df_analysis.copy()

# Ensure AGE is numeric and capture initial unique patient count
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
initial_patients = df['OR_CASE_ID'].nunique()

# Step A: Filter for 18+
df = df[df["AGE"] >= 18].copy()
under_18_removed = initial_patients - df['OR_CASE_ID'].nunique()
post_pediatric_count = df['OR_CASE_ID'].nunique()

# Step B: Restrict to ages < 90 years
df = df[df["AGE"] < 90].copy()
over_90_removed = post_pediatric_count - df['OR_CASE_ID'].nunique()

# Report cohort attrition for reproducibility
print(f"--- Attrition Report ---")
print(f"Initial unique patients: {initial_patients:,}")
print(f"Removed pediatric cases (<18): {under_18_removed:,}")
print(f"Removed geriatric outliers (>=90): {over_90_removed:,}")
print(f"Final analytic cohort (18-89): {df['OR_CASE_ID'].nunique():,}")
print("-" * 25)

# Standardize Sex (0=Male, 1=Female)
if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins (Note: upper bound 90 ensures 85-89 group is captured)
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())

## Fit adjusted exposure and dose models

This section defines the Eleveld age-adjusted pharmacodynamic reference function, prepares the regression dataset, creates prespecified age strata, and fits spline-based models for peak modelled effect-site concentration and weight-normalised induction dose.

In [ ]:
# Eleveld Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# 1) PREPARATION (18-89 Only)
# ----------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()

if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

bins   = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

df_reg = df.dropna(subset=["Max_Effect_Concentration", "AGE", "ASA_SCORE", "BMI", "SEX"]).copy()

# 2) MODELING
# ------------
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
formula_ce = f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_ce = smf.ols(formula_ce, data=df_reg).fit()

g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Predictions
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid = model_ce.get_prediction(grid_df).summary_frame()

bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_ce = model_ce.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()



In [ ]:
# 1) MODELING DOSE
# -----------------
formula_dose = f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_dose = smf.ols(formula_dose, data=df_reg).fit()

pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()



In [ ]:
# 1. SETUP & STANDARDIZED MODELS
# ---------------------------------------------------------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()
if df["SEX"].dtype == "object": df["SEX"] = df["SEX"].map({"M":0,"F":1,"m":0,"f":1})

bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

# Fit Models
AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()

# 2. CALCULATE NORMALISED % DECAY
# ---------------------------------------------------------------------------
g_means = df[["ASA_SCORE", "BMI", "SEX"]].mean()
bin_stats = df.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
pred_df = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_means[0], BMI=g_means[1], SEX=g_means[2])

# Predictions with CIs
ce_res = model_ce.get_prediction(pred_df).summary_frame()
dose_res = model_dose.get_prediction(pred_df).summary_frame()
target_ce50 = 3.08 * np.exp(-0.00635 * (bin_stats["mean_age"] - 35))

# Normalised to 18-24 group
b_ce, b_dose, b_target = ce_res['mean'].iloc[0], dose_res['mean'].iloc[0], target_ce50.iloc[0]

bin_stats["Dose_Pct"] = (dose_res['mean'] / b_dose) * 100
bin_stats["Ce_Pct"] = (ce_res['mean'] / b_ce) * 100
bin_stats["Ce_L"], bin_stats["Ce_U"] = (ce_res['mean_ci_lower'] / b_ce) * 100, (ce_res['mean_ci_upper'] / b_ce) * 100
bin_stats["Target_Pct"] = (target_ce50 / b_target) * 100



## Calculating the technique-preserving required dose ($D_{\mathrm{req}}$)

### Mathematical logic
The proportional dose reduction required to match the benchmark is calculated using the approximate linear scaling properties of the PK model:

$$
D_{\mathrm{req}} = \left(\frac{\text{Target } Ce_{50}}{\text{Predicted } C_{e,\max}}\right) \times \text{Observed Dose}
$$

Under this technique-preserving approach, the total administered dose is rescaled while the observed pattern of drug delivery, including dose fractionation and infusion timing, is maintained.

In [ ]:
# Calculate model-predicted required dose
# Calculate the continuous theoretical dose required to track the Ce50 target
# D_req = (Target / Predicted_Ce) * Actual_Dose
pred_grid["req_dose"] = (ce50_eleveld(age_grid) / pred_grid["mean"]) * pred_grid_dose["mean"]

# Calculate binned required dose for Panel C and Table normalisation
bin_req_dose = (ce50_eleveld(bin_stats["mean_age"]) / bin_pred_ce["mean"]) * bin_pred_dose["mean"]

# Normalise Required Dose to the 18-24 group baseline (First bin)
# This provides the "4th line" for Panel C
bin_stats["Req_Dose_Pct"] = (bin_req_dose / bin_req_dose.iloc[0]) * 100

#
# This finds the % reduction required compared to the young adult required dose
reduction_at_80 = 100 - bin_stats.loc[bin_stats['Age_Group'] == '75-84', 'Req_Dose_Pct'].values[0]
print(f"Theoretical dose reduction required by age 80: {reduction_at_80:.1f}%")

## Generate the main multidimensional age-exposure figure

This section generates the primary manuscript figure showing modelled peak effect-site concentration, weight-normalised induction dose, and age-adjusted pharmacodynamic requirement across the adult lifespan.

In [ ]:
# --- MASTER FIGURE 1: ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 22))
plt.subplots_adjust(hspace=0.42, top=0.96, bottom=0.06, left=0.12, right=0.95)

# --- PANEL A: EXPOSURE VS BENCHMARK ---
ax1.plot(age_grid, pred_grid["mean"], linewidth=4, color='#1f77b4',
         label="Modelled peak $C_{e,max}$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red',
         label="Age-adjusted $C_{e50}$ benchmark")
# Adjusted predictions (bins)
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.3, edgecolor='#1f77b4')

# Percentage surplus labels
for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.15, f'+{diff:.0f}%',
             ha='center', fontweight='bold', color='red', fontsize=10)

ax1.set_title("A: Modelled Peak Effect-Site Concentration ($C_{e,max}$) vs. Age-Adjusted $C_{e50}$ Benchmark",
             loc='left', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel(r"$\mu\mathrm{g\ ml^{-1}}$", fontsize=12)
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True, fontsize=10)

# --- PANEL B: CLINICIAN DOSE VS MODEL-PREDICTED REQUIREMENT ---
ax2.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c',
         label="Observed clinician dose")
ax2.plot(age_grid, pred_grid["req_dose"], linestyle=":", linewidth=3, color='black',
         label="Model-predicted dose to achieve age-adjusted $C_{e50}$")
# Adjusted predictions (bins)
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.3, edgecolor='#2ca02c')

ax2.set_title("B: Observed Clinician Dose vs. Model-Predicted Dose to Achieve Age-Adjusted $C_{e50}$",
             loc='left', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel(r"Weight-normalised induction dose (mg kg$^{-1}$ ABW)", fontsize=12)
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)
ax2.legend(loc="upper right", frameon=True, fontsize=10)

# --- PANEL C: RELATIVE AGE-ASSOCIATED DECLINE ---
x_indices = np.arange(len(labels))
ax3.plot(x_indices, bin_stats["Dose_Pct"], marker='o', linewidth=3, color='#2ca02c',
         label="Observed dose (% baseline)")
ax3.plot(x_indices, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4',
         label="Modelled $C_{e,max}$ (% baseline)")
ax3.fill_between(x_indices, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)
ax3.plot(x_indices, bin_stats["Target_Pct"], marker='D', linewidth=2, linestyle='--', color='red',
         label="Age-adjusted $C_{e50}$ benchmark (% baseline)")
ax3.plot(x_indices, bin_stats["Req_Dose_Pct"], marker='x', linewidth=3, linestyle=':', color='black',
         label="Model-predicted dose to achieve age-adjusted $C_{e50}$ (% baseline)")

ax3.set_title("C: Relative Age-Associated Decline in Dose, Modelled $C_{e,max}$, and Age-Adjusted Benchmarks",
             loc='left', fontsize=14, fontweight='bold', pad=15)
ax3.set_ylabel("% of Young Adult Baseline (18–24 yr)", fontsize=12)
ax3.set_xlabel("Age Cohort (Years)", fontsize=12)
ax3.set_xticks(x_indices)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.legend(loc='lower left', fontsize=10, frameon=True, ncol=1)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

plt.savefig("figure_final_propofol_divergence.pdf", bbox_inches='tight', dpi=300)
plt.show()

## Generate manuscript tables and age-stratified summaries

This section produces age-stratified adjusted predictions, overshoot estimates, and manuscript-ready table outputs derived from the primary Eleveld analysis.

In [ ]:
# Build manuscript summary table

# 1. MATH: Generate Predictions (From Block 2)
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(
    ASA_SCORE=df_reg["ASA_SCORE"].mean(), BMI=df_reg["BMI"].mean(), SEX=df_reg["SEX"].mean()
)
adj_ce = model_ce.get_prediction(bin_pred_input).summary_frame()
adj_dose = model_dose.get_prediction(bin_pred_input).summary_frame()
targets = ce50_eleveld(bin_stats['mean_age'])
d_req_vals = (targets / adj_ce['mean']) * adj_dose['mean']

# 2. DATASET: Build the Table (From Block 2)
table_exposure = pd.DataFrame({
    'Age_Group': labels,
    'N': [f"{int(x):,}" for x in df_reg.groupby("Age_Group", observed=True).size().values],
    'Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(adj_dose['mean'], adj_dose['mean_ci_lower'], adj_dose['mean_ci_upper'])],
    'D_req': [f"{x:.2f}" for x in d_req_vals],
    'Peak_Ce': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(adj_ce['mean'], adj_ce['mean_ci_lower'], adj_ce['mean_ci_upper'])],
    'Benchmark': [f"{t:.2f}" for t in targets],
    'Surplus': [f"+{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(((adj_ce['mean']/targets)-1)*100, ((adj_ce['mean_ci_lower']/targets)-1)*100, ((adj_ce['mean_ci_upper']/targets)-1)*100)]
})

# 3. FORMATTING: Apply the "Width Fix" (From Block 1)
df_tex = table_exposure.copy()
df_tex.columns = [
    "Age Cohort", "N",
    r"\shortstack{Observed Dose \\ (mg kg$^{-1}$)}",
    r"\shortstack{Predicted $D_{req}$ \\ (mg kg$^{-1}$)}",
    r"\shortstack{Peak $C_e$ \\ ($\mu$g ml$^{-1}$)}",
    r"\shortstack{Benchmark \\ $Ce_{50}$}",
    r"\shortstack{Exposure Surplus \\ (95\% CI)}"
]

# 4. EXPORT: Generate and Save
latex_table = df_tex.to_latex(index=False, escape=False, column_format='l r c c c c r')
final_latex = latex_table.replace('%', r'\%').replace(r'\\%', r'\%')

with open("table_exposure_data.tex", "w") as f:
    f.write(final_latex)

print("✅ Merged table with math and stacked headers saved to table_exposure_data.tex")